In [283]:
from standard import *
from setup import Setup
from market import Market
from functions import *
from chart import *
import plotly.express as px

In [284]:
# Charting

from standard import *
import plotly.graph_objects as go


def chart(logs: pd.DataFrame):

    cols = ["schedule", "hour", "load", "id", "mc", "type", "dispatch"]
    df = logs[cols].copy()

    df = logs.copy()  # schedule,hour,load,id,mc,type,dispatch
    df["t"] = (df["schedule"] - 1) * 24 + df["hour"]

    # 1) Per generator: mc and type (assumed constant per id)
    gen_info = df.groupby("id", as_index=False).agg({"mc": "mean", "type": "first"})

    # 2) Order generators by mc (cheapest first)
    gen_info = gen_info.sort_values("mc")
    # gen_order = gen_info["id"].tolist()

    fig = go.Figure()

    for _, row in gen_info.iterrows():
        gen_id = row["id"]
        gen_type = row["type"]
        g = df[df["id"] == gen_id].sort_values("t")
        fig.add_trace(
            go.Scatter(
                x=g["t"],
                y=g["dispatch"],
                mode="lines",
                stackgroup="one",
                name=f"{gen_type} {gen_id}",  # e.g. "nuclear 25"
                line=dict(width=0.5),
            )
        )

    # Load line
    load = df.drop_duplicates("t")[["t", "load"]].sort_values("t")
    fig.add_trace(
        go.Scatter(
            x=load["t"],
            y=load["load"],
            mode="lines",
            name="Load",
            line=dict(color="black", width=2),
            fill=None,
            stackgroup=None,
        )
    )

    fig.update_layout(
        xaxis_title="Hour",
        yaxis_title="Power (MW)",
    )

    fig.show()


PASTEL_RAINBOW = [
    "#FFB3B3",  # pastel red
    "#FFD9B3",  # pastel orange
    "#FFFBB3",  # pastel yellow
    "#B3FFB3",  # pastel green
    "#B3FFFF",  # pastel cyan
    "#B3D9FF",  # pastel blue
    "#D9B3FF",  # pastel violet
]


def _to_grayscale(hex_color):
    """Convert a hex color to its grayscale equivalent."""
    hex_color = hex_color.lstrip("#")
    r = int(hex_color[0:2], 16)
    g = int(hex_color[2:4], 16)
    b = int(hex_color[4:6], 16)
    # luminance-weighted grayscale
    gray = int(0.299 * r + 0.587 * g + 0.114 * b)
    return f"#{gray:02X}{gray:02X}{gray:02X}"


def chart_by_type(
    logs: pd.DataFrame,
    highlight_ids: list = None,
    same_color_per_type: bool = True,  # NEW OPTION
):

    type_colors = {
        "nuclear": "#72DCC6",
        "solar": "#FDFD96",
        "wind": "#9BCBFB",
        "gas": "#FF6961",
        "coal": "#444141",
        "hydro": "#307CC7",
        "balancing": "#C8A8F8",
    }

    cols = ["schedule", "hour", "load", "id", "mc", "type", "dispatch"]
    df = logs[cols].copy()
    df["t"] = (df["schedule"] - 1) * 24 + df["hour"]

    gen_info = df.groupby("id", as_index=False).agg({"mc": "mean", "type": "first"})
    gen_info = gen_info.sort_values("mc")

    type_counts = gen_info["type"].value_counts().to_dict()

    fig = go.Figure()

    # if using multiple colors per type, keep one rainbow index per type
    # so each tech cycles independently (optional, but nicer)
    per_type_rainbow_idx = {t: 0 for t in type_counts.keys()}
    global_rainbow_idx = 0  # fallback if you prefer global cycling

    for _, row in gen_info.iterrows():
        gen_id = row["id"]
        gen_type = row["type"]
        g = df[df["id"] == gen_id].sort_values("t")

        # --- COLOR LOGIC ---
        if same_color_per_type:
            # all units of the same type use the base type color
            true_color = type_colors.get(gen_type, "#C8C8C8")
            outline = true_color
        else:
            # different colors if multiple units of the same type
            if type_counts.get(gen_type, 0) > 1:
                # choose: per-type or global rainbow indexing
                idx = per_type_rainbow_idx[gen_type]
                true_color = PASTEL_RAINBOW[idx % len(PASTEL_RAINBOW)]
                per_type_rainbow_idx[gen_type] += 1
                outline = true_color
            else:
                true_color = type_colors.get(gen_type, "#C8C8C8")
                outline = true_color

        # grayscale when not highlighted
        if highlight_ids is not None and gen_id not in highlight_ids:
            fill_color = _to_grayscale(true_color)
            line_color = _to_grayscale(outline)
        else:
            fill_color = true_color
            line_color = outline

        fig.add_trace(
            go.Scatter(
                x=g["t"],
                y=g["dispatch"],
                mode="lines",
                stackgroup="one",
                name=f"{gen_type} {gen_id}",
                line=dict(width=0.5, color=line_color),
                fillcolor=fill_color,
            )
        )

    # Load line
    load = df.drop_duplicates("t")[["t", "load"]].sort_values("t")
    fig.add_trace(
        go.Scatter(
            x=load["t"],
            y=load["load"],
            mode="lines",
            name="Load",
            line=dict(color="black", width=2),
            fill=None,
            stackgroup=None,
        )
    )

    fig.update_layout(
        xaxis_title="Hour",
        yaxis_title="Power (MW)",
    )

    fig.show()


In [285]:
solar = setup_solar_bids(1000)
wind = setup_wind_bids(1000)
variable_bids = pd.concat([solar,wind])
st = Setup()

a = [st.units(1, "generator", 30*4, 100*4, i+20, 0, 100*4, 2, True, 0) for i in range(int(50/4))] 
# a += [st.units(1, "balancing", 0, 1000, 100, 0, 5000, 0, True, 0) for _ in range(1)] 
# a += [st.units(1, "balancing", 0, 1000, 100, 0, 5000, 0, True, 0) for _ in range(1)] 


st.setup_generator_bids(a)
mo = Market(st.original_bids, st.load_schedule.iloc[0:24*6*2])
mo.start()
chart(mo.logs)

In [286]:
logs = mo.logs.copy()
logs['op_margin'] = np.where(logs['online'] == True,(logs['mcp'] - logs['mc'])*logs['mcp'],0)
logs['sch_h'] = logs['schedule'].astype(str) + '_' + logs['hour'].astype(str)

In [287]:
g = logs.groupby(['id']).agg(total_dispatch=('dispatch','sum'),mc=('mc','first')).reset_index()
g['full_load'] = 288 * 100 * 4
g['capacity_factor'] = g['total_dispatch'] / g['full_load']
id = 4
g[g['id'] == id]

,id,total_dispatch,mc,full_load,capacity_factor
4,4,"104,276.846",24,115200,0.905


In [288]:
import numpy as np
import pandas as pd

a = logs[logs['id'] == id]

def calculate_capex_from_IRR(opex_monthly, op_profit_288, life_years):
    hours_per_period = 288
    hours_per_year = 8760
    periods_per_year = hours_per_year / hours_per_period

    # Allocate monthly opex to each 288h block
    hours_per_month = hours_per_year / 12.0
    periods_per_month = hours_per_month / hours_per_period
    opex_288 = opex_monthly / periods_per_month

    net_profit_288 = op_profit_288 - opex_288
    n_periods = int(round(periods_per_year * life_years))

    # Range of target annual IRRs (e.g. 2% to 30%)
    irr_annual_grid = np.linspace(0.02, 0.30, 15)  # 15 points between 2% and 30%

    # Convert annual IRR to per-288h IRR
    irr_288_grid = (1 + irr_annual_grid) ** (1 / periods_per_year) - 1

    # Capex formula for each per-288h IRR (annuity present value)
    capex_grid = net_profit_288 * (1 - (1 + irr_288_grid) ** (-n_periods)) / irr_288_grid

    df = pd.DataFrame({
        "irr_annual": irr_annual_grid,
        "irr_annual_pct": irr_annual_grid * 100,
        "irr_288h": irr_288_grid,
        "irr_288h_pct": irr_288_grid * 100,
        "capex": capex_grid,
    })

    return df

# Example usage with your DataFrame `a`
opex_monthly = 2000
op_profit_288 = a["op_margin"].sum()
life_years = 10

df = calculate_capex_from_IRR(opex_monthly, op_profit_288, life_years)
df.head(10)


,irr_annual,irr_annual_pct,irr_288h,irr_288h_pct,capex
0,0.020,2.000,0.001,0.065,"3,297,563.699"
1,0.040,4.000,0.001,0.129,"3,005,939.198"
2,0.060,6.000,0.002,0.192,"2,753,259.692"
3,0.080,8.000,0.003,0.253,"2,533,264.332"
4,0.100,10.000,0.003,0.314,"2,340,832.455"
5,0.120,12.000,0.004,0.373,"2,171,755.113"
6,0.140,14.000,0.004,0.432,"2,022,556.118"
7,0.160,16.000,0.005,0.489,"1,890,351.129"
8,0.180,18.000,0.005,0.546,"1,772,736.118"
9,0.200,20.000,0.006,0.601,"1,667,698.668"
